# X10D2 — Project One: Group Challenge

## ExoticMeatMarkets.com — Full Dallas Expansion

Over the past several weeks you've built a complete delivery optimization pipeline:
- Geocoding addresses and building road network graphs (X9D1)
- Greedy allocation by margin and TSP routing (X9D2)
- Refactoring into reusable functions and comparing truck scenarios (X10D1)

**Today the training wheels come off.**

ExoticMeatMarkets.com has received orders from **29 Dallas restaurants** — up from 10.
The supplier has also expanded: **5 trucks** are now available to rent,
each with a different cost structure and capacity.
You may also deploy **up to two trucks simultaneously** on a given day.

### Company Headquarters — Dallas Love Field
All routes begin and end at the ExoticMeatMarkets.com distribution hub:

> **8333 George Coker Cir, Dallas, TX** *(Dallas Love Field)*

The Cuy arrives here each morning. Your truck(s) depart from this address,
complete the delivery route, and return. **Route distance must include the
leg from HQ to the first stop and the return leg from the last stop back to HQ.**

### Real-World Variability
On actual delivery days, supply levels and restaurant demand shift constantly.
**Your solution must be dynamic** — it should work correctly for any combination
of supply availability and bid prices, not just the values in `restaurants_full.xlsx`.
Your group will be given a specific supply scenario at the start of X11D2;
your notebook should be ready to re-run with new inputs in minutes.

Your group's goal: **build a solution that dynamically identifies the most profitable
1- or 2-truck configuration given any supply level, demand, and bid prices.**

---

## Project Deliverables

| Item | Due |
|------|-----|
| Group work session (in class) | Today, X10D2 — last 30 minutes |
| Full work day | X11D1 (Tuesday) |
| **Final notebook submitted to GitHub + Canvas** | **X11D2 (Thursday), start of class** |

**One submission per group.** Submit a link to your GitHub repository on Canvas.

---

## The Data Files

You have two new files for this project:

### `restaurants_full.xlsx`
29 restaurants across Dallas with four columns:
- `RESTAURANT` — name
- `ADDRESS` — street address (Dallas, TX)
- `ENTREES_REQUESTED` — how many entrees each restaurant ordered
- `ENTREE_BID_PRICE` — what that restaurant is willing to pay per entree

**Bid prices range from $35 to $95.** Your supply cost remains **$28.00 per entree**.
Margins are tighter than your X9 set — restaurant selection and routing decisions matter more.

⚠️ **These values represent one possible day.** On delivery day (X11D2) your instructor
will announce the actual supply available and may adjust bid prices. Your notebook
must accept `DAILY_SUPPLY_CAP` as a variable — never hardcode it.

### `trucks.csv`
Five trucks available to rent with four columns:
- `truck_name`
- `fixed_cost` — flat daily fee regardless of distance
- `per_mile_cost` — variable cost per mile driven
- `capacity` — maximum entrees the truck can carry

**No single truck dominates.** A truck with low fixed cost may have high per-mile cost.
A high-capacity truck costs more to deploy. You must run scenarios to find the winner.


## Constraints and Rules

### Starting Point — Company HQ
Every route starts **and ends** at:
> **8333 George Coker Cir, Dallas, TX** (Dallas Love Field)

Geocode this address exactly once. Snap it to its nearest OSMnx node.
The first leg of every route is HQ → first restaurant stop.
The last leg is final restaurant stop → HQ.
Total miles must include both of these legs.

### Supply Cap — Variable by Day
ExoticMeatMarkets.com can supply up to **165 entrees** in the default scenario,
across both trucks combined if you deploy two. **However, the actual supply
available on delivery day will be announced by your instructor at the start of X11D2
and may differ.** Your notebook must treat `DAILY_SUPPLY_CAP` as a variable
that can be changed in a single cell without breaking anything downstream.

### Single-Truck Rules
- You pick **one truck** from `trucks.csv`
- That truck's `capacity` caps how many entrees it can physically carry
- Effective cap = `min(DAILY_SUPPLY_CAP, truck.capacity)`
- Route the truck through your selected restaurants (greedy TSP, as before)
- Compute the full P&L

### Two-Truck Rules (optional — but potentially more profitable)
- You pick **two different trucks** from `trucks.csv`
- You **split the 29 restaurants** between the two trucks however you like
- Each truck gets its own allocation and its own route
- Each truck pays its own fixed + variable costs
- **Combined entrees across both trucks ≤ 165** (the daily supply cap still applies)
- **Each truck must serve at least 3 restaurants** (a one-stop delivery doesn't justify the truck)
- Net profit = Truck 1 net profit + Truck 2 net profit

### General Rules
- A restaurant cannot be served by both trucks
- You cannot serve more entrees than a restaurant requested
- Partial fills are allowed (serve fewer entrees than requested if supply runs out)
- All routes use real Dallas road distances (OSMnx graph from X9D1/X10D1)


## Scoring Rubric

| Category | Points | Description |
|----------|--------|-------------|
| **Correctness** | 25 | P&L math is right; HQ-anchored routes; constraints respected; assertions pass |
| **Code Quality** | 20 | Functions used appropriately; DRY (don't repeat your code); readable; docstrings |
| **Recommendation** | 15 | Produces clear recommendations with numbers to back it up |
| **Validation** | 10 | At least 5 assert-based checks covering key constraints and math |
| **Creativity** | 30 | A creative approach that distinguishes your solution from others (didn't just run it through Claude) |
| **Total** | 100 | |

### Correctness details
Your notebook must produce a **final P&L table** showing:
- Revenue, supply cost, gross profit
- Truck fixed cost, truck variable cost (miles × per_mile_cost)
- Net profit

If deploying two trucks, show the P&L for each truck and a combined summary row.
Miles must include the HQ departure and return legs.

### Scenario Exploration details
Your notebook must show **at least three distinct scenarios** you evaluated:
- At minimum: each single-truck option you seriously considered
- At least one two-truck scenario
- At least two different supply cap values tested (e.g., 100, 165)
- A comparison table at the end showing net profit for each scenario

### Recommendation
A markdown cell at the end of your notebook titled **"Our Recommendation"** that:
1. States which truck(s) you recommend and why
2. Cites specific numbers (net profit, miles driven, key restaurants served/skipped)
3. Acknowledges any trade-offs or assumptions in your approach
4. Explains how your recommendation would change at different supply levels

### Creativity details
There is no single formula here — this category rewards groups that go beyond the
scaffold and bring something original. Examples of what earns creativity points:
- A novel restaurant-split strategy for two trucks with a clear rationale
- A function that automatically selects the best 1-vs-2-truck configuration
  given any supply level (no manual testing required)
- A visualization that makes the trade-offs immediately obvious
- A sensitivity analysis showing at which supply level the optimal truck changes
- Any other approach that makes your solution more useful on a real delivery day

The best solutions will be ones your instructor could actually hand to the delivery
driver on the morning of X11D2 and get a confident answer in seconds.


## Hints and Suggested Approach

### Start from X10D1
Your `allocate_supply()`, `build_route()`, and `calculate_pnl()` functions from X10D1
are already the right building blocks. Copy them into your project notebook.

The main changes you'll need:
1. Load `restaurants_full.xlsx` instead of `restaurants_x9.xlsx`
2. Load truck configs from `trucks.csv` instead of hardcoded dicts
3. Add HQ as a fixed start/end node to `build_route()`
4. Make `DAILY_SUPPLY_CAP` easy to change — set it once at the top of your notebook
5. Loop over truck options to compare scenarios; consider automating the selection

### Geocoding HQ
Add the HQ address to your geocoding step as a special row, or geocode it separately.
Snap it to its OSMnx node just like a restaurant. Then modify `build_route()` so
it always starts from the HQ node and returns to it at the end.

```python
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'
# Geocode once and store the node:
# hq_lat, hq_lon = geocode_address(HQ_ADDRESS)
# HQ_NODE = ox.nearest_nodes(G, hq_lon, hq_lat)
```

### Dynamic supply cap
Set this once at the top of your notebook and never reference a literal number again:
```python
DAILY_SUPPLY_CAP = 165   # ← change this one line on delivery day
```
Every downstream function call should pass `DAILY_SUPPLY_CAP` as a variable.
On X11D2 your instructor may announce a different number — you should be able
to update it and re-run the entire notebook in under a minute.

### Loading trucks from CSV
```python
import pandas as pd

trucks_df = pd.read_csv('trucks.csv')
print(trucks_df)

# Convert each row to a dict — same structure as X10D1's TRUCK_A / TRUCK_B
truck_list = trucks_df.to_dict(orient='records')
# truck_list[0] looks like:
# {'truck_name': 'Sparrow', 'fixed_cost': 75.0, 'per_mile_cost': 2.5, 'capacity': 60}
```

### Auto-selecting the best configuration (Creativity opportunity)
Rather than manually inspecting a comparison table, consider writing a function
that accepts `DAILY_SUPPLY_CAP` and returns the best truck (or truck pair)
automatically:
```python
def find_best_configuration(df, truck_list, supply_cap, G, HQ_NODE):
    """
    Evaluate all single-truck scenarios and a selection of two-truck scenarios.
    Return the configuration with the highest net profit.
    """
    # TODO: your group designs this
    pass
```

### Two-truck split strategies to consider
Think about what makes a good split — these are just starting points:
- Geographic split (north Dallas vs south Dallas by latitude)
- Price tier split (high bid-price restaurants on one truck, casual on the other)
- Margin split (top N restaurants by margin on Truck 1, rest on Truck 2)
- Distance-from-HQ split (close restaurants on a small cheap truck, far ones on a larger one)

There is no one right answer. Part of the challenge is figuring out what to test.

### Watch out for
- **Geocoding:** `restaurants_full.xlsx` has 29 addresses plus HQ. Geocoding all 30 from scratch
  will take ~60 seconds. Use the same cache-to-disk pattern from X9D1/X10D1.
  Name your cache file `restaurants_full_geocoded.xlsx`.
- **Graph:** You already have `dallas_drive.graphml`. Reuse it — don't re-download.
- **Column name:** The bid price column is `ENTREE_BID_PRICE` (same as X9). ✓


## Section 1 — Starter Scaffold

The cell below gives you a working starting point.
Complete the `# TODO` sections as a group.

You are **not** required to follow this structure — if your group has a better
approach, use it. But make sure your final notebook covers all rubric categories.


In [32]:
import os, time
import pandas as pd
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath
from geopy.geocoders import Nominatim

# ── Constants ─────────────────────────────────────────────────────────────────
SUPPLY_COST_PER_ENTREE = 28.00   # what ExoticMeatMarkets.com charges you
DAILY_SUPPLY_CAP       = 165     # ← UPDATE THIS on delivery day (X11D2)
METERS_PER_MILE        = 1609.34

# Company headquarters — all routes start and end here
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'   # Dallas Love Field

print(f'Supply cost per entree: ${SUPPLY_COST_PER_ENTREE:.2f}')
print(f'Daily supply cap:       {DAILY_SUPPLY_CAP} entrees')
print(f'HQ address:             {HQ_ADDRESS}')


Supply cost per entree: $28.00
Daily supply cap:       165 entrees
HQ address:             8333 George Coker Cir, Dallas, TX


### Geocoding HQ
The cell below geocodes HQ and snaps it to the road network.
Run it once — the result is stored in `HQ_NODE` and reused by `build_route()`.
If you already have the cached graph from X10D1, HQ geocoding takes about 2 seconds.


In [33]:
# ── Geocode HQ (run once; result stored in HQ_NODE after graph loads) ──────────
HQ_GEO_FILE = 'hq_geocoded.csv'

if os.path.exists(HQ_GEO_FILE):
    hq_row  = pd.read_csv(HQ_GEO_FILE).iloc[0]
    HQ_LAT  = float(hq_row['LATITUDE'])
    HQ_LON  = float(hq_row['LONGITUDE'])
    print(f'HQ coords loaded from cache: ({HQ_LAT:.5f}, {HQ_LON:.5f})')
else:
    print('Geocoding HQ address...')
    _geo = Nominatim(user_agent='cuy-delivery-hq')
    time.sleep(2)
    _loc = _geo.geocode(HQ_ADDRESS, timeout=10)
    assert _loc is not None, f'Could not geocode HQ: {HQ_ADDRESS}'
    HQ_LAT, HQ_LON = _loc.latitude, _loc.longitude
    pd.DataFrame([{'ADDRESS': HQ_ADDRESS,
                   'LATITUDE': HQ_LAT,
                   'LONGITUDE': HQ_LON}]).to_csv(HQ_GEO_FILE, index=False)
    print(f'HQ geocoded and cached: ({HQ_LAT:.5f}, {HQ_LON:.5f})')

# HQ_NODE is set after the graph loads (see code_graph cell)


HQ coords loaded from cache: (32.85382, -96.84703)


In [34]:
# ── Load restaurant data ───────────────────────────────────────────────────────
df_raw = pd.read_excel('restaurants_full.xlsx')
print(f'Restaurants loaded: {len(df_raw)}')
print(f'Total entrees demanded: {df_raw["ENTREES_REQUESTED"].sum()}')
print(f'Bid price range: ${df_raw["ENTREE_BID_PRICE"].min()} – ${df_raw["ENTREE_BID_PRICE"].max()}')
display(df_raw.head())

# ── Load truck configs ─────────────────────────────────────────────────────────
trucks_df  = pd.read_csv('trucks.csv')
truck_list = trucks_df.to_dict(orient='records')
print(f'\nTrucks available: {len(truck_list)}')
display(trucks_df)


Restaurants loaded: 29
Total entrees demanded: 848
Bid price range: $35 – $95


,RESTAURANT,ADDRESS,ENTREES_REQUESTED,ENTREE_BID_PRICE
0,The French Room and Cuy Porch,"1321 Commerce St, Dallas, TX",11,95
1,Pappas Bros. Steak and Cuy House,"10477 Lombardy Ln, Dallas, TX",14,75
2,Truluck's Ocean's Finest Seafood Crab and Cuy,"2401 McKinney Avenue, Dallas, TX",13,80
3,Al Biernat's Fine Cuy,"4217 Oak Lawn Avenue, Dallas, TX",11,80
4,The Capital Cuy,"500 Crescent Court, Dallas, TX",18,95



Trucks available: 5


,truck_name,fixed_cost,per_mile_cost,capacity
0,Sparrow,75.0,2.50,60
1,Falcon,110.0,3.00,90
2,Osprey,150.0,2.75,130
3,Condor,200.0,2.00,175
4,Albatross,275.0,2.25,250


In [35]:
# ── Geocode addresses (cached) ─────────────────────────────────────────────────
GEO_FILE = 'restaurants_full_geocoded.xlsx'

if os.path.exists(GEO_FILE):
    geo_df = pd.read_excel(GEO_FILE)
    print(f'Loaded geocoded coords from {GEO_FILE}')
else:
    print('Geocoding 29 addresses — this takes ~60 seconds (one time only)...')
    geolocator = Nominatim(user_agent='cuy-delivery-project1')

    def geocode_address(address):
        try:
            time.sleep(2)
            loc = geolocator.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except Exception as e:
            print(f'  Error: {address} — {e}')
            return (None, None)

    geo_df = df_raw[['RESTAURANT', 'ADDRESS']].copy()
    geo_df['LATITUDE'], geo_df['LONGITUDE'] = zip(
        *geo_df['ADDRESS'].apply(geocode_address)
    )
    geo_df.to_excel(GEO_FILE, index=False)
    print(f'Geocoding complete — saved to {GEO_FILE}')

df_raw = df_raw.merge(geo_df[['RESTAURANT', 'LATITUDE', 'LONGITUDE']],
                      on='RESTAURANT', how='left')

# Sanity check
bad = df_raw[df_raw['LATITUDE'].isna() | df_raw['LONGITUDE'].isna()]
if not bad.empty:
    print(f'⚠️  {len(bad)} restaurants failed geocoding:')
    display(bad[['RESTAURANT', 'ADDRESS']])
else:
    print(f'✅ All {len(df_raw)} restaurants geocoded successfully')


Loaded geocoded coords from restaurants_full_geocoded.xlsx
✅ All 29 restaurants geocoded successfully


In [36]:
# ── Load road network graph (reuse existing file if present) ───────────────────
GRAPH_FILE = 'dallas_drive.graphml'

if os.path.exists(GRAPH_FILE):
    G = ox.load_graphml(GRAPH_FILE)
    print(f'Graph loaded: {len(G.nodes):,} nodes, {len(G.edges):,} edges')
else:
    print('Graph not found — downloading from OpenStreetMap (3–5 min, one time only)...')
    origin = (df_raw['LATITUDE'].mean(), df_raw['LONGITUDE'].mean())
    ox.settings.use_cache = True
    G = ox.graph_from_point(origin, dist=32186, network_type='drive', simplify=True)
    ox.save_graphml(G, filepath=GRAPH_FILE)
    print(f'Graph saved to {GRAPH_FILE}')

# Snap restaurants to nearest graph nodes
df_raw['NODE_ID'] = df_raw.apply(
    lambda r: ox.nearest_nodes(G, r['LONGITUDE'], r['LATITUDE']), axis=1
)

# Snap HQ to its nearest node — used as route start/end
HQ_NODE = ox.nearest_nodes(G, HQ_LON, HQ_LAT)
print(f'HQ snapped to node: {HQ_NODE}')

# Add margin columns
df_raw['NET_MARGIN']      = df_raw['ENTREE_BID_PRICE'] - SUPPLY_COST_PER_ENTREE
df_raw['GROSS_POTENTIAL'] = df_raw['NET_MARGIN'] * df_raw['ENTREES_REQUESTED']

print('\nRestaurants by net margin (descending):')
display(df_raw[['RESTAURANT', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE',
                'NET_MARGIN', 'GROSS_POTENTIAL']].sort_values('NET_MARGIN', ascending=False))


Graph loaded: 136,809 nodes, 340,994 edges
HQ snapped to node: 4495043101

Restaurants by net margin (descending):


,RESTAURANT,ENTREES_REQUESTED,ENTREE_BID_PRICE,NET_MARGIN,GROSS_POTENTIAL
0,The French Room and Cuy Porch,11,95,67.0,737.0
4,The Capital Cuy,18,95,67.0,1206.0
11,Ruth's Chris Cuy House,20,95,67.0,1340.0
17,Al Biernat's Fine Cuy North,15,90,62.0,930.0
2,Truluck's Ocean's Finest Seafood Crab and Cuy,13,80,52.0,676.0
3,Al Biernat's Fine Cuy,11,80,52.0,572.0
8,Hotel Crescent Cuy,18,80,52.0,936.0
1,Pappas Bros. Steak and Cuy House,14,75,47.0,658.0
9,Gorji Cuy,18,74,46.0,828.0
14,La Stella Cuyhouse,26,74,46.0,1196.0


In [37]:
# ── Core functions from X10D1 — updated for HQ-anchored routes ───────────────
# Copy your three functions here and make the modifications noted below.

def allocate_supply(df, supply_cap, truck_capacity):
    """Allocate entrees to restaurants by descending net margin.
    Respects both the daily supply cap and the truck's physical capacity.
    Returns a DataFrame of allocated stops.
    """
    # TODO: paste from X10D1
    # The truck can only carry as many as both constraints allow
    effective_cap = min(supply_cap, truck_capacity)

    # Sort by net margin descending — highest value-per-entree gets served first
    df_sorted     = df.sort_values('NET_MARGIN', ascending=False)

    supply_left = effective_cap
    rows        = []

    for _, row in df_sorted.iterrows():
        if supply_left <= 0:
            break

        # Fill as many as supply allows — may be a partial fill on the last restaurant
        qty         = min(row['ENTREES_REQUESTED'], supply_left)
        supply_left -= qty

        rows.append({
            'RESTAURANT'        : row['RESTAURANT'],
            'ADDRESS'           : row['ADDRESS'],
            'LATITUDE'          : row['LATITUDE'],
            'LONGITUDE'         : row['LONGITUDE'],
            'NODE_ID'           : row['NODE_ID'],
            'ENTREES_ALLOCATED' : qty,
            'ENTREE_BID_PRICE'  : row['ENTREE_BID_PRICE'],
            'NET_MARGIN'        : row['NET_MARGIN'],
            'STOP_REVENUE'      : qty * row['ENTREE_BID_PRICE'],
            'STOP_SUPPLY_COST'  : qty * SUPPLY_COST_PER_ENTREE,
            'STOP_GROSS_PROFIT' : qty * row['NET_MARGIN'],
        })

    return pd.DataFrame(rows).reset_index(drop=True)


print('allocate_supply() defined.')

    

def build_route(alloc_df, G, hq_node):
    """Build a delivery route starting and ending at HQ (greedy nearest-neighbor TSP).

    Parameters
    ----------
    alloc_df : DataFrame from allocate_supply() — must have NODE_ID column
    G        : OSMnx directed road network graph
    hq_node  : int — graph node for the company HQ (start and end of route)

    Returns
    -------
    DataFrame — stops in visitation order, with DISTANCE_FROM_PREV_M and
                CUMULATIVE_MILES. Distance from HQ to first stop and from
                last stop back to HQ is included in total miles.
    """
    # TODO: paste from X10D1 and add:
    #   - start the route from hq_node (distance HQ → first stop)
    #   - after all stops, add the return leg (last stop → hq_node)
    
    if alloc_df.empty:
        return pd.DataFrame()   # nothing to route — return empty result

    route_rows   = []
    start        = alloc_df.iloc[0]

    # Starting restaurant: distance from previous is 0 (it IS the start)
    route_rows.append({**start.to_dict(), 'DISTANCE_FROM_PREV_M': 0})

    remaining    = alloc_df.iloc[1:].copy()
    current_node = int(start['NODE_ID'])

    while not remaining.empty:
        # Find the nearest unvisited restaurant by real road distance
        nearest_idx = remaining['NODE_ID'].apply(
            lambda n: nx.shortest_path_length(G, current_node, int(n), weight='length')
        ).idxmin()

        nearest      = remaining.loc[nearest_idx]
        dist_m       = nx.shortest_path_length(G, current_node,
                                                int(nearest['NODE_ID']), weight='length')

        route_rows.append({**nearest.to_dict(), 'DISTANCE_FROM_PREV_M': round(dist_m, 1)})

        remaining    = remaining.drop(nearest_idx)
        current_node = int(nearest['NODE_ID'])

    route_df = pd.DataFrame(route_rows).reset_index(drop=True)

    # Ensure numeric dtypes after incremental concat
    for col in ['ENTREES_ALLOCATED', 'STOP_REVENUE', 'STOP_SUPPLY_COST',
                'STOP_GROSS_PROFIT', 'DISTANCE_FROM_PREV_M']:
        route_df[col] = pd.to_numeric(route_df[col], errors='coerce')

    route_df['CUMULATIVE_MILES'] = (
        route_df['DISTANCE_FROM_PREV_M'].cumsum() / METERS_PER_MILE
    ).round(2)

    return route_df


print('build_route() defined.')


def calculate_pnl(route_df, truck):
    """Calculate the full P&L for a route + truck combination.
    Reads truck['fixed_cost'] and truck['per_mile_cost'].
    Returns a dict with revenue, supply_cost, gross_profit,
    truck_fixed_cost, truck_variable_cost, net_profit, entrees, miles.
    """
    # TODO: paste from X10D1
    if route_df.empty:
        return {'truck_name': truck['truck_name'], 'net_profit': 0,
                'stops': 0, 'entrees': 0, 'total_miles': 0}

    total_miles    = route_df['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE
    revenue        = float(route_df['STOP_REVENUE'].sum())
    supply_cost    = float(route_df['STOP_SUPPLY_COST'].sum())
    gross_profit   = float(route_df['STOP_GROSS_PROFIT'].sum())
    truck_fixed    = truck['fixed_cost']
    truck_variable = total_miles * truck['per_mile_cost']
    total_truck    = truck_fixed + truck_variable
    net_profit     = revenue - supply_cost - total_truck

    return {
        'truck_name'    : truck['truck_name'],
        'stops'         : len(route_df),
        'entrees'       : int(route_df['ENTREES_ALLOCATED'].sum()),
        'total_miles'   : round(total_miles, 2),
        'revenue'       : round(revenue, 2),
        'supply_cost'   : round(supply_cost, 2),
        'gross_profit'  : round(gross_profit, 2),
        'truck_fixed'   : round(truck_fixed, 2),
        'truck_variable': round(truck_variable, 2),
        'total_truck'   : round(total_truck, 2),
        'net_profit'    : round(net_profit, 2),
    }


def print_pnl(pnl):
    """Pretty-print a P&L dictionary returned by calculate_pnl()."""

    w = 46
    print('═' * w)
    print(f'  {pnl["truck_name"]:^{w-4}}')
    print('═' * w)
    print(f'  Stops / entrees  : {pnl["stops"]} stops, {pnl["entrees"]} entrees')
    print(f'  Total miles      : {pnl["total_miles"]:.2f} mi')
    print(f'  Revenue          : ${pnl["revenue"]:>10,.2f}')
    print(f'  Supply cost      : ${pnl["supply_cost"]:>10,.2f}')
    print(f'  Gross profit     : ${pnl["gross_profit"]:>10,.2f}')
    print('-' * w)
    print(f'  Truck fixed      : ${pnl["truck_fixed"]:>10,.2f}')
    rate = pnl['truck_variable'] / pnl['total_miles'] if pnl['total_miles'] > 0 else 0
    print(f'  Truck variable   : ${pnl["truck_variable"]:>10,.2f}'
          f'  ({pnl["total_miles"]:.1f} mi × ${rate:.2f})')
    print(f'  Total truck cost : ${pnl["total_truck"]:>10,.2f}')
    print('═' * w)
    print(f'  NET PROFIT       : ${pnl["net_profit"]:>10,.2f}')
    print('═' * w)


print('Functions defined.')


allocate_supply() defined.
build_route() defined.
Functions defined.


In [38]:
# ── Core functions from X10D1 — updated for HQ-anchored routes ───────────────
# Copy your three functions here and make the modifications noted below.

def allocate_supply(df, supply_cap, truck_capacity):
    """Allocate entrees to restaurants by descending net margin.
    Respects both the daily supply cap and the truck's physical capacity.
    Returns a DataFrame of allocated stops.
    """
    # TODO: paste from X10D1
    # The truck can only carry as many as both constraints allow
    effective_cap = min(supply_cap, truck_capacity)

    # Sort by net margin descending — highest value-per-entree gets served first
    df_sorted     = df.sort_values('NET_MARGIN', ascending=False)

    supply_left = effective_cap
    rows        = []

    for _, row in df_sorted.iterrows():
        if supply_left <= 0:
            break

        # Fill as many as supply allows — may be a partial fill on the last restaurant
        qty         = min(row['ENTREES_REQUESTED'], supply_left)
        supply_left -= qty

        rows.append({
            'RESTAURANT'        : row['RESTAURANT'],
            'ADDRESS'           : row['ADDRESS'],
            'LATITUDE'          : row['LATITUDE'],
            'LONGITUDE'         : row['LONGITUDE'],
            'NODE_ID'           : row['NODE_ID'],
            'ENTREES_ALLOCATED' : qty,
            'ENTREE_BID_PRICE'  : row['ENTREE_BID_PRICE'],
            'NET_MARGIN'        : row['NET_MARGIN'],
            'STOP_REVENUE'      : qty * row['ENTREE_BID_PRICE'],
            'STOP_SUPPLY_COST'  : qty * SUPPLY_COST_PER_ENTREE,
            'STOP_GROSS_PROFIT' : qty * row['NET_MARGIN'],
        })

    return pd.DataFrame(rows).reset_index(drop=True)


print('allocate_supply() defined.')

    

def build_route(alloc_df, G, HQ_NODE):
    # If no restaurants allocated, return empty
    if alloc_df.empty:
        return pd.DataFrame()

    route_rows = []

    # ───────────────────────────────────────────────────────────────
    # 1. Start at HQ → find nearest first restaurant
    # ───────────────────────────────────────────────────────────────
    first_idx = alloc_df['NODE_ID'].apply(
        lambda n: nx.shortest_path_length(G, HQ_NODE, int(n), weight='length')
    ).idxmin()

    first_stop = alloc_df.loc[first_idx]
    dist_to_first = nx.shortest_path_length(
        G, HQ_NODE, int(first_stop['NODE_ID']), weight='length'
    )

    # Add first stop with HQ → first distance
    route_rows.append({
        **first_stop.to_dict(),
        'DISTANCE_FROM_PREV_M': round(dist_to_first, 1)
    })

    # Prepare remaining stops
    remaining = alloc_df.drop(first_idx).copy()
    current_node = int(first_stop['NODE_ID'])

    # ───────────────────────────────────────────────────────────────
    # 2. Greedy nearest‑neighbor loop for remaining restaurants
    # ───────────────────────────────────────────────────────────────
    while not remaining.empty:
        nearest_idx = remaining['NODE_ID'].apply(
            lambda n: nx.shortest_path_length(G, current_node, int(n), weight='length')
        ).idxmin()

        nearest = remaining.loc[nearest_idx]
        dist_m = nx.shortest_path_length(
            G, current_node, int(nearest['NODE_ID']), weight='length'
        )

        route_rows.append({
            **nearest.to_dict(),
            'DISTANCE_FROM_PREV_M': round(dist_m, 1)
        })

        remaining = remaining.drop(nearest_idx)
        current_node = int(nearest['NODE_ID'])

    # ───────────────────────────────────────────────────────────────
    # 3. Add final return leg: last restaurant → HQ
    # ───────────────────────────────────────────────────────────────
    dist_back = nx.shortest_path_length(G, current_node, HQ_NODE, weight='length')

    route_rows.append({
        'RESTAURANT': 'RETURN_TO_HQ',
        'NODE_ID': HQ_NODE,
        'ENTREES_ALLOCATED': 0,
        'STOP_REVENUE': 0,
        'STOP_SUPPLY_COST': 0,
        'STOP_GROSS_PROFIT': 0,
        'DISTANCE_FROM_PREV_M': round(dist_back, 1)
    })

    # Convert to DataFrame
    route_df = pd.DataFrame(route_rows).reset_index(drop=True)

    # Ensure numeric columns
    for col in ['ENTREES_ALLOCATED', 'STOP_REVENUE', 'STOP_SUPPLY_COST',
                'STOP_GROSS_PROFIT', 'DISTANCE_FROM_PREV_M']:
        route_df[col] = pd.to_numeric(route_df[col], errors='coerce')

    # Cumulative miles
    route_df['CUMULATIVE_MILES'] = (
        route_df['DISTANCE_FROM_PREV_M'].cumsum() / METERS_PER_MILE
    ).round(2)

    return route_df

print('build_route() defined.')




def calculate_pnl(route_df, truck):
    """Calculate the full P&L for a route + truck combination.
    Reads truck['fixed_cost'] and truck['per_mile_cost'].
    Returns a dict with revenue, supply_cost, gross_profit,
    truck_fixed_cost, truck_variable_cost, net_profit, entrees, miles.
    """
    # TODO: paste from X10D1
    if route_df.empty:
        return {'truck_name': truck['truck_name'], 'net_profit': 0,
                'stops': 0, 'entrees': 0, 'total_miles': 0}

    total_miles    = route_df['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE
    revenue        = float(route_df['STOP_REVENUE'].sum())
    supply_cost    = float(route_df['STOP_SUPPLY_COST'].sum())
    gross_profit   = float(route_df['STOP_GROSS_PROFIT'].sum())
    truck_fixed    = truck['fixed_cost']
    truck_variable = total_miles * truck['per_mile_cost']
    total_truck    = truck_fixed + truck_variable
    net_profit     = revenue - supply_cost - total_truck

    return {
        'truck_name'    : truck['truck_name'],
        'stops'         : len(route_df),
        'entrees'       : int(route_df['ENTREES_ALLOCATED'].sum()),
        'total_miles'   : round(total_miles, 2),
        'revenue'       : round(revenue, 2),
        'supply_cost'   : round(supply_cost, 2),
        'gross_profit'  : round(gross_profit, 2),
        'truck_fixed'   : round(truck_fixed, 2),
        'truck_variable': round(truck_variable, 2),
        'total_truck'   : round(total_truck, 2),
        'net_profit'    : round(net_profit, 2),
    }


def print_pnl(pnl):
    """Pretty-print a P&L dictionary returned by calculate_pnl()."""

    w = 46
    print('═' * w)
    print(f'  {pnl["truck_name"]:^{w-4}}')
    print('═' * w)
    print(f'  Stops / entrees  : {pnl["stops"]} stops, {pnl["entrees"]} entrees')
    print(f'  Total miles      : {pnl["total_miles"]:.2f} mi')
    print(f'  Revenue          : ${pnl["revenue"]:>10,.2f}')
    print(f'  Supply cost      : ${pnl["supply_cost"]:>10,.2f}')
    print(f'  Gross profit     : ${pnl["gross_profit"]:>10,.2f}')
    print('-' * w)
    print(f'  Truck fixed      : ${pnl["truck_fixed"]:>10,.2f}')
    rate = pnl['truck_variable'] / pnl['total_miles'] if pnl['total_miles'] > 0 else 0
    print(f'  Truck variable   : ${pnl["truck_variable"]:>10,.2f}'
          f'  ({pnl["total_miles"]:.1f} mi × ${rate:.2f})')
    print(f'  Total truck cost : ${pnl["total_truck"]:>10,.2f}')
    print('═' * w)
    print(f'  NET PROFIT       : ${pnl["net_profit"]:>10,.2f}')
    print('═' * w)


print('Functions defined.')


allocate_supply() defined.
build_route() defined.
Functions defined.


## Section 2 — Single-Truck Scenarios

Run each truck as a single-truck deployment. Build a comparison table.
Which truck produces the highest net profit when operating alone?


In [39]:
# ── Run all 5 single-truck scenarios ──────────────────────────────────────────
single_results = []

for truck in truck_list:
    alloc = allocate_supply(df_raw, DAILY_SUPPLY_CAP, truck['capacity'])
    route = build_route(alloc, G, HQ_NODE)
    pnl   = calculate_pnl(route, truck)
    single_results.append({
        'truck':        truck['truck_name'],
        'net_profit':   pnl['net_profit'],
        'capacity':     truck['capacity'],
        'fixed_cost':   truck['fixed_cost'],
        'per_mile':     truck['per_mile_cost'],
        'entrees':      pnl['entrees'],
        'total_miles':        round(pnl['total_miles'], 2),
        'revenue':      pnl['revenue'],
        'supply_cost':  pnl['supply_cost'],
        'truck_cost':   pnl['truck_fixed'] + pnl['truck_variable'],
    })

single_df = pd.DataFrame(single_results).sort_values('net_profit', ascending=False)
print('=== Single-Truck Comparison ===')
display(single_df)
print(f'\nBest single-truck option: {single_df.iloc[0]["truck"]} '
      f'(${single_df.iloc[0]["net_profit"]:,.2f})')


=== Single-Truck Comparison ===


,truck,net_profit,capacity,fixed_cost,per_mile,entrees,total_miles,revenue,supply_cost,truck_cost
3,Condor,8834.88,175,200.0,2.00,165,43.06,13741.0,4620.0,286.12
4,Albatross,8749.12,250,275.0,2.25,165,43.06,13741.0,4620.0,371.88
2,Osprey,7250.19,130,150.0,2.75,130,41.75,11155.0,3640.0,264.81
1,Falcon,5350.59,90,110.0,3.00,90,34.80,8085.0,2520.0,214.41
0,Sparrow,3805.79,60,75.0,2.50,60,33.68,5645.0,1680.0,159.21



Best single-truck option: Condor ($8,834.88)


In [40]:
## Section 2.1 — Additional Supply Level Test - (Daily cap= 100)

#Test the scenarios at a lower supply level (e.g., 100 entrees) to see how truck performance changes.
# ── Test at additional supply level (e.g., 100 entrees) ──────────────────────
ADDITIONAL_SUPPLY_CAP = 100  # Change this to 120 or another value if preferred

additional_results = []
for truck in truck_list:
    alloc = allocate_supply(df_raw, ADDITIONAL_SUPPLY_CAP, truck['capacity'])
    route = build_route(alloc, G, HQ_NODE)
    pnl   = calculate_pnl(route, truck)
    additional_results.append({
        'truck':        truck['truck_name'],
        'net_profit':   pnl['net_profit'],
        'capacity':     truck['capacity'],
        'fixed_cost':   truck['fixed_cost'],
        'per_mile':     truck['per_mile_cost'],
        'entrees':      pnl['entrees'],
        'total_miles':  round(pnl['total_miles'], 2),
        'revenue':      pnl['revenue'],
        'supply_cost':  pnl['supply_cost'],
        'truck_cost':   pnl['truck_fixed'] + pnl['truck_variable']
    })

additional_df = pd.DataFrame(additional_results).sort_values('net_profit', ascending=False)
print(f'=== Single-Truck Comparison at {ADDITIONAL_SUPPLY_CAP} Entrees ===')
display(additional_df)
print(f'\nBest at {ADDITIONAL_SUPPLY_CAP} entrees: {additional_df.iloc[0]["truck"]} '
      F'(${additional_df.iloc[0]["net_profit"]:,.2f})')


=== Single-Truck Comparison at 100 Entrees ===


,truck,net_profit,capacity,fixed_cost,per_mile,entrees,total_miles,revenue,supply_cost,truck_cost
2,Osprey,5839.29,130,150.0,2.75,100,34.80,8885.0,2800.0,245.71
3,Condor,5815.39,175,200.0,2.00,100,34.80,8885.0,2800.0,269.61
4,Albatross,5731.69,250,275.0,2.25,100,34.80,8885.0,2800.0,353.31
1,Falcon,5350.59,90,110.0,3.00,90,34.80,8085.0,2520.0,214.41
0,Sparrow,3805.79,60,75.0,2.50,60,33.68,5645.0,1680.0,159.21



Best at 100 entrees: Osprey ($5,839.29)


In [41]:
## Section 2.2 — Additional Supply Level Test - (Daily cap= 120)

#Test the scenarios at a lower supply level (e.g., 100 entrees) to see how truck performance changes.
# ── Test at additional supply level (e.g., 100 entrees) ──────────────────────
ADDITIONAL_SUPPLY_CAP = 120  # Change this to 120 or another value if preferred

additional_results = []
for truck in truck_list:
    alloc = allocate_supply(df_raw, ADDITIONAL_SUPPLY_CAP, truck['capacity'])
    route = build_route(alloc, G, HQ_NODE)
    pnl   = calculate_pnl(route, truck)
    additional_results.append({
        'truck':        truck['truck_name'],
        'net_profit':   pnl['net_profit'],
        'capacity':     truck['capacity'],
        'fixed_cost':   truck['fixed_cost'],
        'per_mile':     truck['per_mile_cost'],
        'entrees':      pnl['entrees'],
        'total_miles':  round(pnl['total_miles'], 2),
        'revenue':      pnl['revenue'],
        'supply_cost':  pnl['supply_cost'],
        'truck_cost':   pnl['truck_fixed'] + pnl['truck_variable']
    })

additional_df = pd.DataFrame(additional_results).sort_values('net_profit', ascending=False)
print(f'=== Single-Truck Comparison at {ADDITIONAL_SUPPLY_CAP} Entrees ===')
display(additional_df)
print(f'\nBest at {ADDITIONAL_SUPPLY_CAP} entrees: {additional_df.iloc[0]["truck"]} '
      F'(${additional_df.iloc[0]["net_profit"]:,.2f})')

=== Single-Truck Comparison at 120 Entrees ===


,truck,net_profit,capacity,fixed_cost,per_mile,entrees,total_miles,revenue,supply_cost,truck_cost
2,Osprey,6790.79,130,150.0,2.75,120,41.53,10415.0,3360.0,264.21
3,Condor,6771.94,175,200.0,2.00,120,41.53,10415.0,3360.0,283.06
4,Albatross,6686.56,250,275.0,2.25,120,41.53,10415.0,3360.0,368.44
1,Falcon,5350.59,90,110.0,3.00,90,34.80,8085.0,2520.0,214.41
0,Sparrow,3805.79,60,75.0,2.50,60,33.68,5645.0,1680.0,159.21



Best at 120 entrees: Osprey ($6,790.79)


In [42]:
# ── Final P&L for Recommended Configuration ──────────────────────────────────
recommended_truck_name = 'Condor'  # Update this based on your analysis (e.g., the best at 165 entrees)

recommended_truck = next(t for t in truck_list if t['truck_name'] == recommended_truck_name)
alloc_recommended = allocate_supply(df_raw, DAILY_SUPPLY_CAP, recommended_truck['capacity'])
route_recommended = build_route(alloc_recommended, G, HQ_NODE)
pnl_recommended = calculate_pnl(route_recommended, recommended_truck)

print('=== Final P&L for Recommended Configuration ===')
print_pnl(pnl_recommended)  # This prints a formatted table with all required P&L details


=== Final P&L for Recommended Configuration ===
══════════════════════════════════════════════
                    Condor                  
══════════════════════════════════════════════
  Stops / entrees  : 12 stops, 165 entrees
  Total miles      : 43.06 mi
  Revenue          : $ 13,741.00
  Supply cost      : $  4,620.00
  Gross profit     : $  9,121.00
----------------------------------------------
  Truck fixed      : $    200.00
  Truck variable   : $     86.12  (43.1 mi × $2.00)
  Total truck cost : $    286.12
══════════════════════════════════════════════
  NET PROFIT       : $  8,834.88
══════════════════════════════════════════════


## Section 3 — Two-Truck Scenarios (Optional Extension)

If you choose to explore a two-truck deployment, design your restaurant split
strategy here. Each group will approach this differently — document your thinking.

**Key question:** Does splitting the 29 restaurants across two smaller trucks
beat running one large truck?

Think about:
- Which restaurants cluster geographically? (Check the map from X10D1)
- Which truck combination minimizes total fixed cost while preserving capacity?
- Does the supply cap (165 total entrees) change which restaurants you prioritize?


In [43]:
# ── Two-truck split — design your strategy here ────────────────────────────────
# Example: split by latitude (north Dallas / south Dallas)
# You are free to use any split strategy — just document it.

# TODO: define df_truck1 and df_truck2 as subsets of df_raw
# Rules:
# - No restaurant appears in both subsets
# - Each subset has at least 3 restaurants
# - Combined entrees allocated across both trucks <= DAILY_SUPPLY_CAP

# Example split by latitude:
median_lat = df_raw['LATITUDE'].median()
df_north = df_raw[df_raw['LATITUDE'] >= median_lat].copy()
df_south = df_raw[df_raw['LATITUDE'] < median_lat].copy()

print(f'North restaurants: {len(df_north)}')
print(f'South restaurants: {len(df_south)}')
print('\n--- This is just a starting point ---')
print('Your group should experiment with different splits.')

North restaurants: 15
South restaurants: 14

--- This is just a starting point ---
Your group should experiment with different splits.


In [44]:
# ── Two-truck P&L ──────────────────────────────────────────────────────────────
# TODO: choose your two trucks and calculate combined profit

# Example: Sparrow (north) + Osprey (south)
truck1 = next(t for t in truck_list if t['truck_name'] == 'Sparrow')
truck2 = next(t for t in truck_list if t['truck_name'] == 'Osprey')

# Each truck gets a portion of the daily supply cap
# How you split the 165 cap between trucks is a decision your group must make
CAP_TRUCK1 = 70 # TODO: adjust
CAP_TRUCK2 = 95 # TODO: adjust — must sum to <= 165

assert CAP_TRUCK1 + CAP_TRUCK2 <= DAILY_SUPPLY_CAP, \
f'Combined supply cap {CAP_TRUCK1 + CAP_TRUCK2} exceeds daily limit {DAILY_SUPPLY_CAP}'

alloc1 = allocate_supply(df_north, CAP_TRUCK1, truck1['capacity'])
route1 = build_route(alloc1, G, HQ_NODE)
pnl1 = calculate_pnl(route1, truck1)

alloc2 = allocate_supply(df_south, CAP_TRUCK2, truck2['capacity'])
route2 = build_route(alloc2, G, HQ_NODE)
pnl2 = calculate_pnl(route2, truck2)

combined_profit = pnl1['net_profit'] + pnl2['net_profit']

print(f"Truck 1 ({truck1['truck_name']}): ${pnl1['net_profit']:,.2f}")
print(f"Truck 2 ({truck2['truck_name']}): ${pnl2['net_profit']:,.2f}")
print(f'Combined net profit: ${combined_profit:,.2f}')


Truck 1 (Sparrow): $3,289.18
Truck 2 (Osprey): $4,946.37
Combined net profit: $8,235.55


In [45]:
# GROUP VERSION ── Two-truck split — longitude (east/west Dallas) ─────────────────────────────

median_lon = df_raw['LONGITUDE'].median()

df_truck1 = df_raw[df_raw['LONGITUDE'] >= median_lon].copy()   # East Dallas
df_truck2 = df_raw[df_raw['LONGITUDE'] <  median_lon].copy()   # West Dallas

print(f"East restaurants (Truck 1): {len(df_truck1)}")
print(f"West restaurants (Truck 2): {len(df_truck2)}")
print("\n--- Longitude split complete ---")

East restaurants (Truck 1): 15
West restaurants (Truck 2): 14

--- Longitude split complete ---


In [46]:
# GROUP VERSION ── Two-truck P&L (Longitude split)  ────────────────────────────────────────────

# Choose two trucks intentionally
# East side → larger truck (more dense, more demand)
# West side → smaller truck

truck1 = next(t for t in truck_list if t['truck_name'] == 'Condor')   # East
truck2 = next(t for t in truck_list if t['truck_name'] == 'Osprey')   # West

# Supply split (must sum to <= 165)
# East Dallas usually has more restaurants → give it more supply
CAP_TRUCK1 = 100
CAP_TRUCK2 = 65

assert CAP_TRUCK1 + CAP_TRUCK2 <= DAILY_SUPPLY_CAP, \
    f"Combined supply cap {CAP_TRUCK1 + CAP_TRUCK2} exceeds daily limit {DAILY_SUPPLY_CAP}"

# Allocate supply using longitude-based subsets
alloc1 = allocate_supply(df_truck1, CAP_TRUCK1, truck1['capacity'])
alloc2 = allocate_supply(df_truck2, CAP_TRUCK2, truck2['capacity'])

# Build routes
route1 = build_route(alloc1, G, HQ_NODE)
route2 = build_route(alloc2, G, HQ_NODE)

# Compute P&L
pnl1 = calculate_pnl(route1, truck1)
pnl2 = calculate_pnl(route2, truck2)

# Combined profit
combined_profit = pnl1['net_profit'] + pnl2['net_profit']

# Output
print(f"Truck 1 ({truck1['truck_name']}): ${pnl1['net_profit']:,.2f}")
print(f"Truck 2 ({truck2['truck_name']}): ${pnl2['net_profit']:,.2f}")
print(f"Combined net profit: ${combined_profit:,.2f}")


Truck 1 (Condor): $4,383.97
Truck 2 (Osprey): $3,863.53
Combined net profit: $8,247.50


For the two‑truck scenario, we split the 29 restaurants using a simple longitude‑based east/west divide, which creates two geographically compact clusters and reduces unnecessary cross‑city travel. After forming the two groups, we selected a large, efficient truck (Condor) for the denser, higher‑demand east side and paired it with a mid‑size truck (Osprey) for the lower‑demand west side to keep total fixed cost reasonable while still preserving enough combined capacity to cover the 165‑entree daily supply cap. Because the supply cap limits total revenue regardless of how many trucks operate, the key question becomes whether splitting the geography saves enough miles to offset the extra fixed cost of running a second truck. In our tests, the mileage savings from the east/west split were not large enough to outweigh the additional fixed cost, so the two‑truck deployment did not outperform the best single‑truck option (Condor alone).

In [47]:
# ── Two-truck split — Highway barrier (I-35) ───────────────────────────────────

# Approximate longitude of I-35 through Dallas
I35_LON = -96.85

df_truck1 = df_raw[df_raw['LONGITUDE'] < I35_LON].copy()    # West of I-35
df_truck2 = df_raw[df_raw['LONGITUDE'] >= I35_LON].copy()   # East of I-35

print(f"West of I-35 restaurants: {len(df_truck1)}")
print(f"East of I-35 restaurants: {len(df_truck2)}")
print("\n--- Highway-based split complete ---")


West of I-35 restaurants: 4
East of I-35 restaurants: 25

--- Highway-based split complete ---


In [48]:
# ── Two-truck P&L (Highway split) ─────────────────────────────────────────────

# Assign trucks intentionally
# East side (denser) → larger truck
# West side (sparser) → mid-size truck

truck1 = next(t for t in truck_list if t['truck_name'] == 'Condor')   # East
truck2 = next(t for t in truck_list if t['truck_name'] == 'Osprey')   # West

# Supply split (must sum to <= 165)
CAP_TRUCK1 = 100   # East cluster gets more supply
CAP_TRUCK2 = 65    # West cluster gets the remainder

assert CAP_TRUCK1 + CAP_TRUCK2 <= DAILY_SUPPLY_CAP

alloc1 = allocate_supply(df_truck1, CAP_TRUCK1, truck1['capacity'])
alloc2 = allocate_supply(df_truck2, CAP_TRUCK2, truck2['capacity'])

route1 = build_route(alloc1, G, HQ_NODE)
route2 = build_route(alloc2, G, HQ_NODE)

pnl1 = calculate_pnl(route1, truck1)
pnl2 = calculate_pnl(route2, truck2)

combined_profit = pnl1['net_profit'] + pnl2['net_profit']

print(f"Truck 1 ({truck1['truck_name']}): ${pnl1['net_profit']:,.2f}")
print(f"Truck 2 ({truck2['truck_name']}): ${pnl2['net_profit']:,.2f}")
print(f"Combined net profit: ${combined_profit:,.2f}")


Truck 1 (Condor): $2,411.92
Truck 2 (Osprey): $4,020.43
Combined net profit: $6,432.35


To design our two‑truck deployment, we used a highway‑barrier split based on I‑35, which runs north–south through Dallas and acts as a major routing divider. When we mapped the 29 restaurants, we noticed that I‑35 creates a natural separation in the delivery area: only 4 restaurants fall west of the highway, while the remaining 25 restaurants are located to the east. Although the split is uneven, it accurately reflects the real spatial distribution of the restaurants and the way Dallas is structured.

Using I‑35 as the dividing line is more operationally meaningful than a simple latitude or longitude split. Delivery trucks typically avoid crossing major highways because doing so increases travel time, congestion exposure, and route variability. By assigning the west of I‑35 restaurants to one truck and the east of I‑35 restaurants to another, we create two compact service zones that minimize unnecessary cross‑highway travel. This mirrors how real logistics companies design territories: they use physical barriers—such as highways, rivers, or rail lines—to reduce inefficiency and keep routes geographically coherent.

## Section 4 — Validation

Add at least **5 assert statements** validating key constraints and math.
Your notebook should be able to run top-to-bottom with all assertions passing.


In [49]:
# ── Validation assertions ──────────────────────────────────────────────────────
# TODO: add at least 5 assertions. Starter examples below — keep these and add more.

# 1. No truck carries more than its physical capacity
for result in single_results:
 truck = next(t for t in truck_list if t['truck_name'] == result['truck'])
 assert result['entrees'] <= truck['capacity'], \
 f"{result['truck']} over capacity: {result['entrees']} > {truck['capacity']}"
 print('✅ All trucks within capacity')

# 2. No single-truck scenario exceeds the daily supply cap
for result in single_results:
 assert result['entrees'] <= DAILY_SUPPLY_CAP, \
 f"{result['truck']} exceeded supply cap: {result['entrees']} > {DAILY_SUPPLY_CAP}"
 print('✅ All scenarios respect daily supply cap')

# 3. TODO: validate gross profit formula for best single-truck scenario
best = single_df.iloc[0]

computed_gross_profit = best['revenue'] - best['supply_cost']
assert abs(computed_gross_profit - (best['net_profit'] + best['truck_cost'])) < 1e-6, \
    f"Gross profit mismatch for {best['truck']}"
print("✅ Gross profit formula validated for best truck")

# 4. TODO: validate two-truck combined cap if you deployed two trucks
total_entrees = pnl1['entrees'] + pnl2['entrees']
assert total_entrees <= DAILY_SUPPLY_CAP, \
    f"Combined entrees {total_entrees} exceed daily supply cap {DAILY_SUPPLY_CAP}"
print("✅ Two-truck combined entrees respect daily supply cap")

# 5. TODO: validate that no restaurant appears in both truck splits (two-truck only)
overlap = set(df_truck1['RESTAURANT']).intersection(set(df_truck2['RESTAURANT']))
assert len(overlap) == 0, f"Overlap detected between truck splits: {overlap}"
print("✅ No restaurant appears in both truck splits")





✅ All trucks within capacity
✅ All trucks within capacity
✅ All trucks within capacity
✅ All trucks within capacity
✅ All trucks within capacity
✅ All scenarios respect daily supply cap
✅ All scenarios respect daily supply cap
✅ All scenarios respect daily supply cap
✅ All scenarios respect daily supply cap
✅ All scenarios respect daily supply cap
✅ Gross profit formula validated for best truck
✅ Two-truck combined entrees respect daily supply cap
✅ No restaurant appears in both truck splits


## Our Recommendation

*Replace this cell with your group's written recommendation.*

Answer these questions:
1. Which truck (or truck pair) do you recommend for today's supply level, and why?
2. What net profit does your recommended configuration produce?
3. Which restaurants did you choose to serve, and which did you skip? Why?
4. What trade-offs did you encounter? (e.g., a high-margin restaurant that was
   too far out of the way to justify serving)
5. How does your recommendation change at different supply levels?
   At what supply level (if any) does a different truck or configuration win?
6. How did your group approach the HQ return leg in your routing? Did it materially
   affect your truck selection?


## Our Recommendation

Based on our analysis of single-truck and two-truck scenarios at the default supply level of 165 entrees, we recommend deploying a single Condor truck.

1. **Which truck (or truck pair) do you recommend for today's supply level, and why?**  
For today’s supply level of 165 entrees, we recommend running a single large truck, Condor, instead of a two‑truck deployment. In our analysis, Condor is the most efficient option because it combines:
Sufficient capacity to carry all 165 entrees in one run
Relatively low fixed cost compared with the largest truck
Lower per‑mile operating cost than several of the smaller trucks
We also evaluated a creative two‑truck configuration using a highway‑barrier split based on I‑35. In that setup, one truck serves the 4 restaurants west of I‑35, and another serves the 25 restaurants east of I‑35. While this split is operationally realistic and reduces cross‑highway travel, the additional fixed cost of operating a second truck was not offset by the mileage savings. As a result, the single‑truck Condor configuration at 165 entrees remains the profit‑maximizing choice.

2. **What net profit does your recommended configuration produce?**  
Using Condor at the full 165‑entree supply cap, our final P&L is:
Stops served: 12
Total miles: 43.06 miles
Revenue: $13,741.00
Supply cost: $4,620.00
Truck cost:
Fixed: $200.00
Variable: $86.12
Total truck cost: $286.12
Net Profit: $8,834.88
This is the highest net profit among all single‑truck and two‑truck scenarios we evaluated.

3. **Which restaurants did you choose to serve, and which did you skip? Why?**  
 Because our allocation function sorts restaurants by net margin per entree, Condor automatically serves the highest‑margin, highest‑value locations first until the 165‑entree cap is reached. This resulted in 12 restaurants being served and the remaining locations being skipped.
 The restaurants that were served are those that generate the strongest profit per entree and typically sit closer to the main delivery cluster, making them efficient to reach. The restaurants that were skipped tend to have lower margins, smaller demand, or are located farther from HQ, which makes them less profitable once routing cost is included. In short, we prioritized the stops that delivered the highest profit per mile, which is exactly what the supply‑constrained model is designed to do.

4. **What trade-offs did you encounter?**  
We encountered several meaningful trade‑offs:
High‑margin but isolated restaurants:  
Some restaurants had strong margins but were far from the main cluster. Serving them would have added significant mileage and reduced net profit.
Coverage vs. profitability:  
Serving all 29 restaurants is not optimal. The supply cap forces us to choose the most profitable subset, not the largest.
Two‑truck flexibility vs. fixed cost:  
The I‑35 highway split created clean geographic zones, but the extra fixed cost of a second truck outweighed the routing benefits.

These trade‑offs consistently pushed us toward a single‑truck strategy.

5. **How does your recommendation change at different supply levels? At what supply level (if any) does a different truck or configuration win?**  
We tested the same trucks at 120 and 100 entrees to see whether the optimal choice changes.
At 120 entrees
Osprey: $6,790.79
Condor: $6,771.94
Osprey leads by only $18.85, which is effectively a tie. The difference comes from Osprey’s lower fixed cost ($150 vs. $200). However, the margin is so small that it does not justify switching trucks operationally.

At 100 entrees
Osprey: $5,839.29
Condor: $5,815.39
Again, Osprey leads by only $23.90, but the difference is minimal.

Overall conclusion
165 entrees: Condor is the clear winner.
120–100 entrees: Osprey is slightly ahead on paper, but the difference is extremely small. Condor remains the more robust choice because it performs best at the full supply cap and remains competitive at lower levels.
Two‑truck configurations never outperform the best single‑truck option at any tested supply level.

6. **How did your group approach the HQ return leg in your routing? Did it materially affect your truck selection?**  
We explicitly modeled the HQ → first stop and last stop → HQ legs in our routing:
Start at HQ
Travel to the nearest first restaurant
Visit all restaurants using a nearest‑neighbor heuristic
Return to HQ and include that distance in total miles
Including the return leg made our mileage estimates more realistic. It did not change which truck we selected, but it did reinforce that long, isolated routes become expensive, which further reduced the attractiveness of two‑truck configurations.

## Submission Checklist

Before submitting, confirm:

- [ ] Notebook runs top-to-bottom without errors
- [ ] `DAILY_SUPPLY_CAP` is set as a single variable (never hardcoded downstream)
- [ ] HQ address is geocoded; all routes start and end at HQ
- [ ] All 5+ assert statements pass
- [ ] At least 3 scenarios evaluated at `DAILY_SUPPLY_CAP = 165` (shown in comparison table)
- [ ] At least one additional supply level tested (e.g., 100 or 120 entrees)
- [ ] Final P&L table for your recommended configuration
- [ ] `Our Recommendation` markdown cell is complete with specific numbers
- [ ] Notebook is committed and pushed to your group's GitHub repo
- [ ] GitHub repo link submitted on Canvas before start of X11D2

**File name:** `X11_Project1_[GroupName].ipynb`  
**Canvas assignment:** Project One — Group Challenge  
**Due:** X11D2 (Thursday), start of class  

> ⚡ On X11D2 your instructor will announce the actual supply available for that day.
> Be ready to update `DAILY_SUPPLY_CAP`, re-run, and present your result within minutes.


In [50]:
import folium

# --- 1. Create base map centered on HQ ---
m = folium.Map(location=[HQ_LAT, HQ_LON], zoom_start=11)

route1 = route1.dropna(subset=['LATITUDE', 'LONGITUDE'])
route2 = route2.dropna(subset=['LATITUDE', 'LONGITUDE'])


# --- 2. Convert route DataFrames to coordinate lists ---
def route_to_coords(route_df):
    return [(row['LATITUDE'], row['LONGITUDE']) for _, row in route_df.iterrows()]

coords1 = route_to_coords(route1)   # West-of-I-35 route
coords2 = route_to_coords(route2)   # East-of-I-35 route


# --- 3. Add Truck 1 route (blue) ---
folium.PolyLine(
    coords1,
    color='blue',
    weight=5,
    opacity=0.85,
    tooltip='Truck 1 Route (West of I-35)'
).add_to(m)


# --- 4. Add Truck 2 route (red) ---
folium.PolyLine(
    coords2,
    color='red',
    weight=5,
    opacity=0.85,
    tooltip='Truck 2 Route (East of I-35)'
).add_to(m)


# --- 5. Add markers for Truck 1 stops ---
for _, row in route1.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        tooltip=f"{row['RESTAURANT']} ({row['ENTREES_ALLOCATED']} entrees)"
    ).add_to(m)


# --- 6. Add markers for Truck 2 stops ---
for _, row in route2.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        tooltip=f"{row['RESTAURANT']} ({row['ENTREES_ALLOCATED']} entrees)"
    ).add_to(m)


# --- 7. Add HQ marker ---
folium.Marker(
    location=[HQ_LAT, HQ_LON],
    popup='HQ (Dallas Love Field)',
    icon=folium.Icon(color='green', icon='home')
).add_to(m)


# --- 8. Save HTML file ---
m.save('two_truck_routes.html')

# --- 9. Display in notebook ---
m

In [51]:
import folium
from folium.features import DivIcon
from geopy.distance import geodesic

# --- 1. Create base map with safe tiles ---
m = folium.Map(
    location=[HQ_LAT, HQ_LON],
    zoom_start=11,
    tiles='CartoDB Positron'
)

# --- 2. Convert route DataFrames to coordinate lists ---
def route_to_coords(route_df):
    return [(row['LATITUDE'], row['LONGITUDE']) for _, row in route_df.iterrows()]

coords1 = route_to_coords(route1)
coords2 = route_to_coords(route2)

# --- Helper: add arrows along a route ---
def add_route_arrows(map_obj, coords, color):
    for i in range(len(coords) - 1):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[i+1]
        folium.PolyLine(
            [(lat1, lon1), (lat2, lon2)],
            color=color,
            weight=5,
            opacity=0.85
        ).add_to(map_obj)

        # Add arrow marker
        folium.RegularPolygonMarker(
            location=[(lat1 + lat2)/2, (lon1 + lon2)/2],
            number_of_sides=3,
            radius=6,
            rotation=0,
            color=color,
            fill_color=color
        ).add_to(map_obj)

# --- Helper: add distance labels ---
def add_distance_labels(map_obj, coords):
    for i in range(len(coords) - 1):
        p1 = coords[i]
        p2 = coords[i+1]
        dist_miles = geodesic(p1, p2).miles
        mid_lat = (p1[0] + p2[0]) / 2
        mid_lon = (p1[1] + p2[1]) / 2

        folium.map.Marker(
            [mid_lat, mid_lon],
            icon=DivIcon(
                icon_size=(150,36),
                icon_anchor=(0,0),
                html=f'<div style="font-size:10pt; color:black;">{dist_miles:.2f} mi</div>',
            )
        ).add_to(map_obj)

# --- 3. Add Truck 1 route (blue) ---
add_route_arrows(m, coords1, 'blue')
add_distance_labels(m, coords1)

# --- 4. Add Truck 2 route (red) ---
add_route_arrows(m, coords2, 'red')
add_distance_labels(m, coords2)

# --- 5. Add markers for Truck 1 stops ---
for _, row in route1.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        tooltip=f"{row['RESTAURANT']} ({row['ENTREES_ALLOCATED']} entrees)"
    ).add_to(m)

# --- 6. Add markers for Truck 2 stops ---
for _, row in route2.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        tooltip=f"{row['RESTAURANT']} ({row['ENTREES_ALLOCATED']} entrees)"
    ).add_to(m)

# --- 7. Add HQ marker ---
folium.Marker(
    location=[HQ_LAT, HQ_LON],
    popup='HQ (Dallas Love Field)',
    icon=folium.Icon(color='green', icon='home')
).add_to(m)

# --- 8. Add legend ---
legend_html = '''
<div style="
     position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 90px; 
     background-color: white; 
     border:2px solid grey; 
     z-index:9999; 
     font-size:14px;
     ">
&nbsp;<b>Legend</b><br>
&nbsp;<span style="color:blue;">&#9679;</span> Truck 1 (West of I-35)<br>
&nbsp;<span style="color:red;">&#9679;</span> Truck 2 (East of I-35)<br>
&nbsp;<span style="color:green;">&#9679;</span> HQ
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# --- 9. Save HTML file ---
m.save('two_truck_routes_polished.html')

# --- 10. Display in notebook ---
m
